In [3]:
# -*- coding: utf-8 -*-
"""
Instagram 캡션 형태소 상관/군집 분석 (샐러드_600)
- 입력  : instagram_captions_샐러드_600.csv
- 열    : ko_tokens_all (문서별 형태소 리스트; 문자열 또는 리스트)
- 산출(모두 utf-8-sig):
  1) morph_stats_샐러드600.csv                : 토큰별 df/tf/중심성/군집/랭킹
  2) morph_clusters_샐러드600.csv             : 군집별 요약(대표 토큰)
  3) morph_edges_npmi_top_샐러드600.csv      : 상위 NPMI 엣지(공동출현 문서수 필터 반영)
  4) tableau_nodes_샐러드600.csv             : Tableau 노드 파일(토큰 특성)
  5) tableau_edges_full_샐러드600.csv        : Tableau 엣지 파일(쌍 + 양끝 노드 특성 결합)
  6) npmi_heatmap_top25_샐러드600.png
  7) top_pairs_npmi_bar_샐러드600.png
  8) cluster_scatter_svd_샐러드600.png
  9) rank_vs_df_scatter_샐러드600.png
"""

import os, re, ast, math, warnings, platform, time
from pathlib import Path
from collections import Counter
import numpy as np
import pandas as pd

from sklearn.decomposition import TruncatedSVD
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

import matplotlib
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

# ================== 경로 ==================
# (당신이 보여준 Win 경로로 기본 세팅)
DATA_DIR = r"C:\Users\sagej\Digital_Pyhton_Study\형태소_영양제\final_cluster"
FNAME    = "kurly_health_merged_20250922_2109 "   # 끝 공백 주의
FNAME    = FNAME.strip()                          # 자동 보정

IN_PATH  = Path(DATA_DIR) / FNAME
OUT_DIR  = Path(DATA_DIR)
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ================== 파라미터 ==================
MIN_DF       = 5         # 최소 문서수(등장 문서 기준)
TOP_V        = 1400      # 상위 DF 토큰만 사용(속도/안정성)
EMB_DIM      = 80        # PPMI-SVD 임베딩 차원
K_RANGE      = (8, 10)   # KMeans k 후보
SIL_SAMPLE   = 1000       # 실루엣 샘플 수(속도용)
MIN_CO_DOCS  = 5         # 엣지 저장 시 공동출현 최소 문서수
HEAT_TOP     = 25        # 히트맵 표시 토큰 수
PAIR_TOP     = 20        # 상위 연관쌍 바차트 수

# ================== 불용어/필터 ==================
# 일반 조사/접속사/형식어
STOPWORDS = set("""
그리고 그러나 그런데 또한 또는 그래서 등의 즉 및 으로 로 은 는 이 가 을 를 과 와 하고 보다 에서 에게 에 에도 에는 에다가 으니까 면 도 만 까지 뿐 처럼 같은 듯 듯이 것 거 데 수 들 등 더 가장 제일 아주 매우 너무 정말 진짜 그냥 혹시 거의 대부분 여러 각각 모든 아무 이런 그런 저런 어떤 무슨
이다 아니다 하다 되다 같다 있다 없다 되었다 입니다 입니다만 아닌가요 아닌듯 같아요 같습니다
이게 저게 그게 이것 저것 그것 여기 저기 거기 오늘 어제 내일 이번 지난 요즘 방금 지금 현재 이후 이전 먼저 그리고나서 또 다시 계속 좀 매우 더 많이 조금 거의

lunchbox bento 넣고 1스푼 먹고 좋이컵 오늘도 IFF 있는 맛있게 다들 같이 하루 위에 도시락을 넣어 오늘은 많은 이제 중에먹을 위해 정도
따로 제가 다른 있어서 항상 가득 모두 싸서 그대로 하는 있습니다 올려 이렇게 듬뿍 담아 맛이 먹는 하나 먹으면 저는 바로 않은
그래도 다양한 있어 맛있어서 챙겨 되는 가득한 즐거운 오랜만에 역시 완전 싶은 들고 보고 있으니 엄청 
""".split())

# 도메인(식품/도시락/리뷰/쇼핑/인스타) 일반 불용어
STOPWORDS |= {
    "제품","상품","브랜드","성분","구성","구매","구입","판매","할인","행사","이벤트","증정","사은품","옵션","용량","세트","세일","재구매",
    "가격","배송","포장","후기","리뷰","평점","별점","추천","만족","불만","개선","문의","연락","무료","예약","구독",
    "마켓","마켓컬리","컬리","이너컬리","링크","스토리","프로필","댓글","협찬","광고","체험단","쿠폰","맞팔","서이추","인스타","해시태그",
    "맛","향","느낌","간편","간단","건강","다이어트","푸드","메뉴","세트구성","주문","도착","출발","국내산","정품","신상","베스트","인기"
}

STOPWORDS |= {
    "도시락","직장인도시락","점심도시락","도시락은","도시락통","만든","함께","완성","열심히","좋아하는","만들어","제품제공","도시락으로","싸기",
    "도시락이","도시락그램","엄마가","만들었어요","수제도시락","주문","한끼","링크","도시락에","시간","없어서","넣은","화이팅","날씨가","도시락만들기",
    "만들고","메뉴추천","않고","넣어서","맛스타그램","벌써","스푼","인분","했는데","도시락레시피","보내세요","엄마표","일찍","이번주","전에",
    "그럼","간단요리","대신","도시락맛집","도시락추천","보니","사실","실리콘","날씨","내가","마지막","있고","점심메뉴추천","준비했어요","클릭",
    "팬에","다녀와","도시락메뉴추천","사용한","싸고","싸는","아직","얼른","쏘락","일어나","하루도","같아서","나는","댓글","준비","추천드려요","하면",
    "행사","김에","덕분에","만들어서","맞팔","바라요","보내시길","사용","생각보다","시작","이번엔","저희","캐릭터","크게","가는","굽네","근데","나무도시락통",
    "레시피는","맛있더라구요","맛집","분들","슈슈아띠","이번주도","일어나서","좋고","좋더라구요","초간단","나름","냉장고","냉장고에","놀고",
    "달궈진","마무리","매번","밥에","분들이","스텐","어린이집","엄마는","조심하세요","혼자","힘내용","그동안","만원","맨날","어떻게","없고",
    "일을","잠시","제품은","주의","지금","즐겁게","채칼","가방","가능해서","갔다가","기준","담고","되서","뭔가","얼마","오늘부터","요즘은","이상",
    "잔뜩","챙기는","챙길","편해요","행복가득한","ㅋㅋ","ㅠㅠ","가서","광고","구입처는","그나저나","금방","깔끔하게","꺼내","끓여주기","날이",
    "눈이","느낌이","다양하게","당장","두른","마음을","무려","밍키","바랍니다","받은","병원","빠르게","사세","삼각","소중한","시에","안에",
    "않는","않아도","야무지게","어느날은","없어요","오래","올린","용기는","이것저걱","일단","저도","좋다","좋을","준비하고","중간","지니","지니도시락","지니벤또",
    "참고해","챙기고","친구","특히","편하게","하게","하네요","현실","화이팅하세요","후다닥","같아","거기에","계모임체험단","그렇게",
    "기능","나오는","날도","넉넉한","느낌","다녀왔어요","다음","당분간","대충","더누리다","도전","돌돌","됩니다","라는","만들기","맛에","맛으로",
    "맛있을","먹어","모음","무조건","문자","물론","물에","물을","번째","보세요","보온도시락","비비고","빼고","사서","사용이","성공","세상","시원한","싶은데",
    "아니라","어제는","여기에","알록달록","역시하림","옛날","오늘두","오늘뭐먹지","요거","일차","있을","자세한","잘게","조합으로","좋다고","좋아해서","좋은데",
    "챙겨서","챙기기","챙기면","챱챱","추가해서","하나면","풀어","하루를","할인","ㅋㅋㅋㅋ","가능해요","가면","가지고","것도","것이","결국","구성","구워",
    "국내산","국내","국산","기분이","나를","나서","나의","내내","내컵","냅다","넣었어요","넘치는","다니는","당일조리","데워","돌아온","들어가는","때는","라고","라부부",
    "마음으로","만들","말고","맛난","맛있는거","모음집","문의가","바람","부어","분들은","비가","빨리","삼행시를","새로","생산","세척","소통","속이","순으로","쉬시고"
}

STOPWORDS |= {
    "lunchbox","bento","넣고","1스푼","먹고","좋이컵","오늘도","IFF","있는","맛있게","다들","같이","하루","위에","도시락을","넣어","오늘은","많은","이제","중에먹을","위해","정도",
    "따로","제가","다른","있어서","항상","가득","모두","싸서","그대로","하는","있습니다","올려","이렇게","듬뿍","담아","맛이","먹는","하나","먹으면","저는","바로","않은",
    "그래도","다양한","있어","맛있어서","챙겨","되는","가득한","즐거운","오랜만에","역시","완전","싶은","들고","보고","있으니","엄청"
}

# 단위/치수/수량 표기
STOPWORDS |= {
    "ml","l","mg","g","kg","kcal","cal","cm","mm","포","봉","팩","캔","병","박스","개입","팩입","세트","박스입","박스형",
    "대","소","중","특","특대","대용량","소용량","정","분","알","회분","개","총","수량","증가","감소"
}

# 흔한 용언/형용사 어간(형태소 정규화 결과에서 자주 등장하는 빈말)
STOPWORDS |= {
    "좋다","맛있다","먹다","먹기","같다","괜찮다","간편하다","간단하다","자주","자다","되다","하다","이다","아니다",
    "있다","없다","느끼다","보이다","느껴지다","가다","오다","하다가","드리다","드림","올려요","해요","됩니다","했어요","합니다"
}

# 허용 1글자 (의미 보존)
ALLOW_1 = {"밥","면","쌀","빵","죽"}

def keep_token(t: str) -> bool:
    """숫자·불용어·불용 1글자 제거"""
    if not t:
        return False
    if t in STOPWORDS:
        return False
    if t.isdigit():
        return False    # 순수 숫자 제거
    if len(t) == 1 and t not in ALLOW_1:
        return False
    return True

# ================== 유틸 ==================
def parse_tokens(x):
    if isinstance(x, list):
        return x
    if isinstance(x, str):
        s = x.strip()
        # 문자열로 저장된 리스트 처리
        if (s.startswith("[") and s.endswith("]")) or (s.startswith("(") and s.endswith(")")):
            try:
                obj = ast.literal_eval(s)
                if isinstance(obj, (list, tuple)):
                    return [str(t) for t in obj]
            except Exception:
                pass
        # 공백 분리 백업
        return [t for t in re.split(r"\s+", s) if t]
    return []

def clean_tok(t: str):
    t = str(t).strip()
    t = re.sub(r"[^0-9A-Za-z가-힣]+", "", t)  # 한글/영숫자만
    return t

def zscore(x):
    x = np.asarray(x, dtype=float)
    return (x - x.mean()) / (x.std() + 1e-9)

def pick_token_column(df):
    # 가장 가능성 높은 열 추정
    cand = [c for c in df.columns if re.search(r"(ko[_ ]?tokens|token|morph|clean|caption)", c, re.I)]
    return cand[0] if cand else df.columns[0]

# ================== 폰트(한글) ==================
sysname = platform.system().lower()
if "windows" in sysname:
    matplotlib.rcParams["font.family"] = "Malgun Gothic"
elif "darwin" in sysname:
    matplotlib.rcParams["font.family"] = "AppleGothic"
else:
    # 리눅스: 나눔/본고딕 둘 중 설치된 것 사용
    for f in ["NanumGothic", "Noto Sans CJK KR", "DejaVu Sans"]:
        try:
            matplotlib.font_manager.findfont(f, fallback_to_default=False)
            matplotlib.rcParams["font.family"] = f
            break
        except Exception:
            pass
matplotlib.rcParams["axes.unicode_minus"] = False

# ================== 1) 데이터 로드 ==================
print(f"[LOAD] {IN_PATH}")
df_raw = pd.read_csv(IN_PATH, encoding="utf-8-sig")
tok_col = pick_token_column(df_raw)

docs_raw = df_raw[tok_col].apply(parse_tokens).tolist()

docs = []
for toks in docs_raw:
    toks2 = [clean_tok(t) for t in toks if isinstance(t, str)]
    toks2 = [t for t in toks2 if keep_token(t)]
    if toks2:
        docs.append(toks2)

N = len(docs)
print(f"[INFO] docs={N}")

# ================== 2) 어휘 선택(df 기준) & DTM(이진) ==================
df_counter = Counter()
tf_counter = Counter()
for toks in docs:
    df_counter.update(set(toks))
    tf_counter.update(toks)

candidates = [(t, c) for t, c in df_counter.items() if c >= MIN_DF]
candidates.sort(key=lambda x: (-x[1], -tf_counter[x[0]]))
vocab = [t for t, _ in candidates[:TOP_V]]
V = len(vocab)
vidx = {t: i for i, t in enumerate(vocab)}
print(f"[INFO] vocab(V)={V}")

DTM = np.zeros((N, V), dtype=np.uint8)  # binary document-term
for i, toks in enumerate(docs):
    cols = {vidx[t] for t in toks if t in vidx}
    if cols:
        DTM[i, list(cols)] = 1

df_vec = DTM.sum(axis=0).astype(int)
tf_vec = np.array([tf_counter[t] for t in vocab], dtype=int)

# ================== 3) 공출현/확률/PMI/NPMI/PHI ==================
co = DTM.T @ DTM  # 문서 단위 공동출현(대각 포함)
total_docs = float(N)
p_t = df_vec / total_docs
p_xy = co / total_docs

EPS = 1e-12
PMI  = np.log((p_xy + EPS) / (p_t[:, None] * p_t[None, :] + EPS))
NPMI = PMI / (-np.log(p_xy + EPS))
np.fill_diagonal(PMI, 0.0)
np.fill_diagonal(NPMI, 0.0)

# PHI(이진 피어슨 상관)
n11 = co.astype(float)
n1_ = df_vec.astype(float)[:, None]
n_1 = df_vec.astype(float)[None, :]
n00 = N - (n1_ + n_1 - n11)
n10 = n1_ - n11
n01 = n_1 - n11
den = np.sqrt(n1_*(N-n1_)*n_1*(N-n_1)) + 1e-12
PHI = (n11*n00 - n10*n01) / den
np.fill_diagonal(PHI, 0.0)

# ================== 4) PPMI 임베딩 + KMeans 군집 ==================
PPMI = np.maximum(PMI, 0.0)

svd = TruncatedSVD(n_components=min(EMB_DIM, max(10, V-1)), random_state=42)
emb = svd.fit_transform(PPMI)

best = {"k": None, "sil": -1, "labels": None}
k_lo, k_hi = K_RANGE
for k in range(k_lo, k_hi+1):
    km = KMeans(n_clusters=k, n_init=10, random_state=42)
    labels = km.fit_predict(emb)
    # 속도 절약: 샘플 실루엣
    sample_size = min(SIL_SAMPLE, len(emb))
    try:
        sil = silhouette_score(emb, labels, sample_size=sample_size, random_state=42)
    except Exception:
        sil = -1
    if sil > best["sil"]:
        best = {"k": k, "sil": sil, "labels": labels}

labels = best["labels"]
K = best["k"]
print(f"[CLUSTER] k={K}, silhouette={best['sil']:.3f}")

# ================== 5) 군집 중심성 + 랭킹 ==================
NPMI_pos = np.where(NPMI > 0, NPMI, 0.0)

# 군집 내 연결강도 중심성(각 토큰의 군집 내 NPMI 양수 합)
cent = np.zeros(V, dtype=float)
for ci in range(K):
    idx = np.where(labels == ci)[0]
    if len(idx) == 0:
        continue
    sub = NPMI_pos[np.ix_(idx, idx)]
    cent[idx] = (sub.sum(axis=1) - np.diag(sub))

# 추가 특성: 임베딩 제1축(구조적 중요도 근사) — NumPy 2.0 호환(np.ptp)
c1 = emb[:, 0].astype(float)
svd_c1 = (c1 - c1.min()) / (np.ptp(c1) + 1e-12)

# 최종 랭크(가중합: 군집중심성>df>tf>구조축)
rank_score = 0.45*zscore(cent) + 0.30*zscore(df_vec) + 0.15*zscore(tf_vec) + 0.10*zscore(svd_c1)

token_stats = (
    pd.DataFrame({
        "token": vocab,
        "df": df_vec,
        "tf": tf_vec,
        "cluster": labels,
        "cluster_centrality": cent,
        "svd_c1": svd_c1,
        "rank_score": rank_score
    })
    .sort_values(["rank_score", "df"], ascending=[False, False])
    .reset_index(drop=True)
)

# 군집 요약(대표 토큰 30개)
rows = []
for ci in range(K):
    sub = token_stats[token_stats["cluster"] == ci].head(30)
    rows.append({
        "cluster": ci,
        "size": int((labels == ci).sum()),
        "top_tokens": ", ".join(sub["token"].tolist())
    })
clusters_df = pd.DataFrame(rows).sort_values("cluster")

# ================== 6) 상위 엣지(쌍) 테이블 ==================
pairs = np.triu_indices(V, 1)
npmi_vals = NPMI[pairs]
co_vals   = co[pairs]
mask = (npmi_vals > 0) & (co_vals >= MIN_CO_DOCS)
order = np.argsort(-npmi_vals[mask])
i_idx = pairs[0][mask][order]
j_idx = pairs[1][mask][order]

edges_df = pd.DataFrame({
    "token1": [vocab[i] for i in i_idx],
    "token2": [vocab[j] for j in j_idx],
    "co_docs": co_vals[mask][order].astype(int),
    "PMI": PMI[pairs][mask][order],
    "NPMI": npmi_vals[mask][order],
    "PHI": PHI[pairs][mask][order]
})

# ================== 7) Tableau용 파일 ==================
# 노드 테이블
nodes_tbl = token_stats.copy()
nodes_tbl.rename(columns={
    "token":"node",
    "df":"node_df",
    "tf":"node_tf",
    "cluster":"node_cluster",
    "cluster_centrality":"node_centrality",
    "svd_c1":"node_svd_c1",
    "rank_score":"node_rank"
}, inplace=True)

# 엣지 + 양끝 노드 특성 결합(총합 CSV)
left  = nodes_tbl.add_prefix("src_")
right = nodes_tbl.add_prefix("dst_")
left.rename(columns={"src_node":"token1"}, inplace=True)
right.rename(columns={"dst_node":"token2"}, inplace=True)

edges_full = (
    edges_df
    .merge(left,  on="token1", how="left")
    .merge(right, on="token2", how="left")
)

# ================== 안전 저장(utf-8-sig, PermissionError 회피) ==================
def safe_save_csv(df: pd.DataFrame, path: Path):
    try:
        df.to_csv(path, index=False, encoding="utf-8-sig")
    except PermissionError:
        alt = path.with_name(path.stem + f"_{int(time.time())}" + path.suffix)
        df.to_csv(alt, index=False, encoding="utf-8-sig")
        print(f"[WARN] PermissionError로 대체 저장: {alt}")

stats_path       = OUT_DIR / "morph_stats_샐러드600.csv"
clusters_path    = OUT_DIR / "morph_clusters_샐러드600.csv"
edges_path       = OUT_DIR / "morph_edges_npmi_top_샐러드600.csv"
nodes_path       = OUT_DIR / "tableau_nodes_샐러드600.csv"
edges_full_path  = OUT_DIR / "tableau_edges_full_샐러드600.csv"

safe_save_csv(token_stats,   stats_path)
safe_save_csv(clusters_df,   clusters_path)
safe_save_csv(edges_df,      edges_path)
safe_save_csv(nodes_tbl,     nodes_path)
safe_save_csv(edges_full,    edges_full_path)

print("[SAVE]")
print(" -", stats_path)
print(" -", clusters_path)
print(" -", edges_path)
print(" -", nodes_path)
print(" -", edges_full_path)

# ================== 9) 시각화(보강) ==================
def savefig(path, tight=True, dpi=200):
    if tight: plt.tight_layout()
    plt.savefig(path, dpi=dpi)
    plt.close()

# (a) NPMI 히트맵: 상위 토큰
HEAT_TOP = min(HEAT_TOP, len(token_stats))
sel_tokens = token_stats["token"].head(HEAT_TOP).tolist()
sel_idx = [vocab.index(t) for t in sel_tokens]
mat = np.where(NPMI>0, NPMI, 0.0)[np.ix_(sel_idx, sel_idx)]

plt.figure(figsize=(9,7))
plt.imshow(mat, interpolation="nearest")
plt.xticks(range(HEAT_TOP), sel_tokens, rotation=90)
plt.yticks(range(HEAT_TOP), sel_tokens)
plt.title("NPMI Heatmap (Top tokens)")
plt.colorbar()
savefig(OUT_DIR / "npmi_heatmap_top25_샐러드600.png")

# (b) 상위 연관쌍 바차트
tp = edges_df.head(PAIR_TOP)
plt.figure(figsize=(10,7))
y = [f"{a}-{b}" for a,b in zip(tp["token1"], tp["token2"])]
plt.barh(range(len(y)), tp["NPMI"].values)
plt.yticks(range(len(y)), y)
plt.xlabel("NPMI")
plt.title("상위 형태소 연관쌍 (NPMI 기준)")
savefig(OUT_DIR / "top_pairs_npmi_bar_샐러드600.png")

# (c) 임베딩 2D 스캐터(군집색)
plt.figure(figsize=(9,7))
xv, yv = emb[:,0], emb[:,1]
plt.scatter(xv, yv, c=labels, s=22)
# 대표 토큰 라벨(각 군집 랭크 상위 3개)
for ci in range(K):
    sub = token_stats[token_stats["cluster"]==ci].head(3)
    for tok in sub["token"]:
        idx = vocab.index(tok)
        plt.text(xv[idx], yv[idx], tok, fontsize=8)
plt.title(f"SVD Embedding Scatter (k={K})")
plt.xlabel("SVD-1"); plt.ylabel("SVD-2")
savefig(OUT_DIR / "cluster_scatter_svd_샐러드600.png")

# (d) 랭크 vs df 산점도(상위 토큰 라벨)
plt.figure(figsize=(9,7))
plt.scatter(token_stats["df"], token_stats["rank_score"], s=18)
for tok in token_stats.head(20)["token"]:
    ridx = token_stats.index[token_stats["token"]==tok][0]
    plt.text(token_stats.loc[ridx, "df"], token_stats.loc[ridx, "rank_score"], tok, fontsize=8)
plt.xlabel("DF (문서수)")
plt.ylabel("Rank Score")
plt.title("형태소 랭킹 vs 등장 문서수")
savefig(OUT_DIR / "rank_vs_df_scatter_샐러드600.png")

print("[DONE] 모든 결과가 utf-8-sig로 저장되었습니다.")


[LOAD] C:\Users\sagej\Digital_Pyhton_Study\형태소_영양제\instagram_captions_샐러드_600.csv


FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\sagej\\Digital_Pyhton_Study\\형태소_영양제\\instagram_captions_샐러드_600.csv'

In [11]:
# -*- coding: utf-8 -*-
"""
kurly_health 리뷰 텍스트 형태소(토큰) 상관/군집 분석
- 입력  : /mnt/data/kurly_health_merged_20250922_2109.csv (경로 변경 가능)
- 토큰 컬럼 우선순위: ko_tokens_all/tokens/morphs/... 없으면 review_text를 간이 토큰화
- 산출(utf-8-sig):
  1) morph_stats_kurly_health.csv
  2) morph_clusters_kurly_health.csv
  3) morph_edges_npmi_top_kurly_health.csv
  4) tableau_nodes_kurly_health.csv
  5) tableau_edges_full_kurly_health.csv
  6) npmi_heatmap_top25_kurly_health.png
  7) top_pairs_npmi_bar_kurly_health.png
  8) cluster_scatter_svd_kurly_health.png
  9) rank_vs_df_scatter_kurly_health.png
"""

import os, re, math, warnings, platform, time
from pathlib import Path
from collections import Counter
import numpy as np
import pandas as pd
from sklearn.decomposition import TruncatedSVD
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import matplotlib
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

# ====== 경로 ======
IN_PATH = Path("kurly_health_merged_20250922_2109.csv")  # <- 로컬 경로로 바꿔도 됨
OUT_DIR = Path(IN_PATH.parent) / "kurly_health_morph"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ====== 파라미터 (필요 시 조정) ======
TEXT_COL_CANDIDATES = ["ko_tokens_all","tokens","token","morphs","morph_tokens",
                       "clean_text","cleaned_text","content","caption","review_text","review","text","body"]
MIN_DF       = 3
TOP_V        = 3000   # 데이터 크기에 맞춘 상한
EMB_DIM      = 60
K_RANGE      = (6, 9)
SIL_SAMPLE   = 4000
MIN_CO_DOCS  = 5
HEAT_TOP     = 25
PAIR_TOP     = 20

# ================== 불용어/필터 ==================
# 일반 조사/접속사/형식어
STOPWORDS = set("""
그리고 그러나 그런데 또한 또는 그래서 등의 즉 및 으로 로 은 는 이 가 을 를 과 와 하고 보다 에서 에게 에 에도 에는 에다가 으니까 면 도 만 까지 뿐 처럼 같은 듯 듯이 것 거 데 수 들 등 더 가장 제일 아주 매우 너무 정말 진짜 그냥 혹시 거의 대부분 여러 각각 모든 아무 이런 그런 저런 어떤 무슨
이다 아니다 하다 되다 같다 있다 없다 되었다 입니다 입니다만 아닌가요 아닌듯 같아요 같습니다
이게 저게 그게 이것 저것 그것 여기 저기 거기 오늘 어제 내일 이번 지난 요즘 방금 지금 현재 이후 이전 먼저 그리고나서 또 다시 계속 좀 매우 더 많이 조금 거의

lunchbox bento 넣고 1스푼 먹고 좋이컵 오늘도 IFF 있는 맛있게 다들 같이 하루 위에 도시락을 넣어 오늘은 많은 이제 중에먹을 위해 정도
따로 제가 다른 있어서 항상 가득 모두 싸서 그대로 하는 있습니다 올려 이렇게 듬뿍 담아 맛이 먹는 하나 먹으면 저는 바로 않은
그래도 다양한 있어 맛있어서 챙겨 되는 가득한 즐거운 오랜만에 역시 완전 싶은 들고 보고 있으니 엄청
""".split())

# 도메인(식품/도시락/리뷰/쇼핑/인스타) 일반 불용어
STOPWORDS |= {
    "제품","상품","브랜드","성분","구성","구매","구입","판매","할인","행사","이벤트","증정","사은품","옵션","용량","세트","세일","재구매",
    "가격","배송","포장","후기","리뷰","평점","별점","추천","만족","불만","개선","문의","연락","무료","예약","구독",
    "마켓","마켓컬리","컬리","이너컬리","링크","스토리","프로필","댓글","협찬","광고","체험단","쿠폰","맞팔","서이추","인스타","해시태그",
    "맛","향","느낌","간편","간단","건강","다이어트","푸드","메뉴","세트구성","주문","도착","출발","국내산","정품","신상","베스트","인기"
}

STOPWORDS |= {
    "도시락","직장인도시락","점심도시락","도시락은","도시락통","만든","함께","완성","열심히","좋아하는","만들어","제품제공","도시락으로","싸기",
    "도시락이","도시락그램","엄마가","만들었어요","수제도시락","주문","한끼","링크","도시락에","시간","없어서","넣은","화이팅","날씨가","도시락만들기",
    "만들고","메뉴추천","않고","넣어서","맛스타그램","벌써","스푼","인분","했는데","도시락레시피","보내세요","엄마표","일찍","이번주","전에",
    "그럼","간단요리","대신","도시락맛집","도시락추천","보니","사실","실리콘","날씨","내가","마지막","있고","점심메뉴추천","준비했어요","클릭",
    "팬에","다녀와","도시락메뉴추천","사용한","싸고","싸는","아직","얼른","쏘락","일어나","하루도","같아서","나는","댓글","준비","추천드려요","하면",
    "행사","김에","덕분에","만들어서","맞팔","바라요","보내시길","사용","생각보다","시작","이번엔","저희","캐릭터","크게","가는","굽네","근데","나무도시락통",
    "레시피는","맛있더라구요","맛집","분들","슈슈아띠","이번주도","일어나서","좋고","좋더라구요","초간단","나름","냉장고","냉장고에","놀고",
    "달궈진","마무리","매번","밥에","분들이","스텐","어린이집","엄마는","조심하세요","혼자","힘내용","그동안","만원","맨날","어떻게","없고",
    "일을","잠시","제품은","주의","지금","즐겁게","채칼","가방","가능해서","갔다가","기준","담고","되서","뭔가","얼마","오늘부터","요즘은","이상",
    "잔뜩","챙기는","챙길","편해요","행복가득한","ㅋㅋ","ㅠㅠ","가서","광고","구입처는","그나저나","금방","깔끔하게","꺼내","끓여주기","날이",
    "눈이","느낌이","다양하게","당장","두른","마음을","무려","밍키","바랍니다","받은","병원","빠르게","사세","삼각","소중한","시에","안에",
    "않는","않아도","야무지게","어느날은","없어요","오래","올린","용기는","이것저걱","일단","저도","좋다","좋을","준비하고","중간","지니","지니도시락","지니벤또",
    "참고해","챙기고","친구","특히","편하게","하게","하네요","현실","화이팅하세요","후다닥","같아","거기에","계모임체험단","그렇게",
    "기능","나오는","날도","넉넉한","느낌","다녀왔어요","다음","당분간","대충","더누리다","도전","돌돌","됩니다","라는","만들기","맛에","맛으로",
    "맛있을","먹어","모음","무조건","문자","물론","물에","물을","번째","보세요","보온도시락","비비고","빼고","사서","사용이","성공","세상","시원한","싶은데",
    "아니라","어제는","여기에","알록달록","역시하림","옛날","오늘두","오늘뭐먹지","요거","일차","있을","자세한","잘게","조합으로","좋다고","좋아해서","좋은데",
    "챙겨서","챙기기","챙기면","챱챱","추가해서","하나면","풀어","하루를","할인","ㅋㅋㅋㅋ","가능해요","가면","가지고","것도","것이","결국","구성","구워",
    "국내산","국내","국산","기분이","나를","나서","나의","내내","내컵","냅다","넣었어요","넘치는","다니는","당일조리","데워","돌아온","들어가는","때는","라고","라부부",
    "마음으로","만들","말고","맛난","맛있는거","모음집","문의가","바람","부어","분들은","비가","빨리","삼행시를","새로","생산","세척","소통","속이","순으로","쉬시고"
}

STOPWORDS |= {
    "lunchbox","bento","넣고","1스푼","먹고","좋이컵","오늘도","IFF","있는","맛있게","다들","같이","하루","위에","도시락을","넣어","오늘은","많은","이제","중에먹을","위해","정도",
    "따로","제가","다른","있어서","항상","가득","모두","싸서","그대로","하는","있습니다","올려","이렇게","듬뿍","담아","맛이","먹는","하나","먹으면","저는","바로","않은",
    "그래도","다양한","있어","맛있어서","챙겨","되는","가득한","즐거운","오랜만에","역시","완전","싶은","들고","보고","있으니","엄청"
}

# 단위/치수/수량 표기
STOPWORDS |= {
    "ml","l","mg","g","kg","kcal","cal","cm","mm","포","봉","팩","캔","병","박스","개입","팩입","세트","박스입","박스형",
    "대","소","중","특","특대","대용량","소용량","정","분","알","회분","개","총","수량","증가","감소"
}

# 흔한 용언/형용사 어간(형태소 정규화 결과에서 자주 등장하는 빈말)
STOPWORDS |= {
    "좋다","맛있다","먹다","먹기","같다","괜찮다","간편하다","간단하다","자주","자다","되다","하다","이다","아니다",
    "있다","없다","느끼다","보이다","느껴지다","가다","오다","하다가","드리다","드림","올려요","해요","됩니다","했어요","합니다"
}

# 허용 1글자 (의미 보존)
ALLOW_1 = {"밥","면","쌀","빵","죽"}

# ====== Regex-based stopword patterns (형태/활용/변형까지 커버) ======
STOP_REGEX_RAW = [
    # 인사/상투구
    r"^(안녕(하)?세요|감사(합니다|해요)?|공지|이벤트\s?안내)$",

    # 광고/협찬/프로모션
    r"^(광고|협찬|체험단|제공|스폰서|ppl|sponsor(?:ed)?)$",

    # SNS/행동 유도
    r"^(구독|맞팔|서이추|링크|프로필|댓글|디엠|dm|message|메시지|메세지)$",
    r"^(인스타|인스타그램|instagram|.*스타그램)$",

    # 브랜드/캠페인 상투어
    r"^(마켓컬리|컬리|이너컬리|kurly|kurl(?:y)?|market\s?kurly)$",
    r"^(?:all|for|life|better|hello|thanks?|event|sale)s?$",  # 캠페인 영단어

    # 가격/프로모션/유통
    r"^(행사|세일|특가|할인|쿠폰|증정|사은품|이벤트|균일가)$",
    r"^(가격|가성비|정가|할인가|무료|유료)$",
    r"^(배송|도착|출발|오늘출발|예약발송|택배|로젠|우체국)$",

    # 리뷰/만족도/메타 표현
    r"^(리뷰|후기|평점|별점|추천|만족|불만|개선)$",

    # 패키징/부속
    r"^(종이.?백|쇼핑.?백|리본|스티커|테이프|포장)$",

    # 카테고리 일반어
    r"^(메뉴|세트(?:구성)?|구성|옵션|용량|패키지)$",
    r"^도시락[가-힣]*$",                   # 도시락*
    r"^(국내산|국내|국산)$",

    # 시간/감탄/의성어
    r"^(오늘|어제|내일|이번|지난|요즘)[가-힣]*$",
    r"^[ㅋㅎㅠㅜ]+$",

    # 빈말 형용/동사 활용형 (좋다/있다/없다/하다/되다 등 변형)
    r"^(?:좋|맛있|괜찮|간편하|간단하|편하|유용하|깔끔하|빠르|예쁘|부드럽)(?:았|었)?(?:다|요|네요|습니다|해요|합니다)?$",
    r"^(?:있|없|하|되|같)(?:었|았)?(?:다|요|네요|습니다|해요|합니다)?$",

    # ‘맛·향·느낌’류 상투어(어근+접미)
    r".*(?:맛|향|느낌)$",
]



# ===== 확장: 리뷰 상투어/활용형 대량 차단용 공통 종결 =====
COMMON_END = r"(?:다|요|네요|습니다|예요|에요|였어요|었어요|았어요|어|서|로|으로|에서|도록|다는|인데|인데요|인데도|는데|으니|으니까|려고|할려고|기로|기도|니|지만)?"

# 자주 나오는 동사/형용사 어간(이 어간으로 시작하면 대부분 불용어 취급)
STEMS = r"(?:좋|있|없|먹|구매|구입|주문|샀|추천하|받았|쓰|되|마시|않)"

# 위 어간 변형 전부 + 일부 단어(토큰) 정확 매칭
STOP_REGEX_RAW += [
    # 어간 기반: 좋아요/좋아서/좋아/좋은/있도록/있다는/먹고있어요/먹었는데/구매했습니다/주문했어요/샀는데/추천합니다/되어/썼어요/마시고/않아요 등
    rf"^{STEMS}[가-힣]*{COMMON_END}$",

    # 특례: '있도록/있다는/있는데'를 명시 커버(토큰화 방식에 따라 어간 패턴을 못 잡는 경우 대비)
    r"^있(?:도록|다는|는데)$",

    # 특례: 맞춤법 변형/흔한 입력 실수
    r"^할?려고$",            # 할려고 / 하려고
    r"^들어요$",             # 들어요(메타 감상)
    r"^좋아하시네요$",       # 존칭/시제 결합

    # 단일 토큰(정확 매칭)
    r"^(이건|이거|확실히|유용하게|금액채우기용|무엇보다|그런지|성분도)$",
]

# (기존 라인 유지/대체) 정규식 컴파일


STOP_PATTERNS = [re.compile(p, re.IGNORECASE) for p in STOP_REGEX_RAW]


# ====== 유틸 ======
def read_csv_safely(path: Path) -> pd.DataFrame:
    for enc in ["utf-8-sig","utf-8","cp949","euc-kr"]:
        try:
            return pd.read_csv(path, encoding=enc)
        except Exception:
            pass
    return pd.read_csv(path)

def pick_text_column(df: pd.DataFrame) -> str:
    cols = [c for c in TEXT_COL_CANDIDATES if c in df.columns]
    if cols: return cols[0]
    for c in df.columns:
        if re.search(r"(review|text|content|caption)", c, re.I):
            return c
    return df.columns[-1]

def simple_ko_tokenize(s: str):
    if not isinstance(s, str): return []
    s = re.sub(r"[\r\n\t]+", " ", s)
    toks = re.findall(r"[가-힣]{2,}", s)              # 2자 이상 한글
    toks1 = [t for t in re.findall(r"[가-힣]", s) if t in ALLOW_1]  # 의미 1자
    en = [t.lower() for t in re.findall(r"[A-Za-z]{2,}", s)]       # 라틴(옵션)
    return toks + toks1 + en

def keep_token(t: str) -> bool:
    if not t:
        return False
    if t in STOPWORDS:
        return False
    if t.isdigit():
        return False
    if len(t) == 1 and t not in ALLOW_1:
        return False
    # 정규표현식 기반 불용어 필터
    for pat in STOP_PATTERNS:
        if pat.match(t):
            return False
    return True


def zscore(x):
    x = np.asarray(x, dtype=float)
    return (x - x.mean()) / (x.std() + 1e-9)

# ====== 폰트 ======
sysname = platform.system().lower()
if "windows" in sysname:
    matplotlib.rcParams["font.family"] = "Malgun Gothic"
elif "darwin" in sysname:
    matplotlib.rcParams["font.family"] = "AppleGothic"
else:
    for f in ["NanumGothic", "Noto Sans CJK KR", "DejaVu Sans"]:
        try:
            matplotlib.font_manager.findfont(f, fallback_to_default=False)
            matplotlib.rcParams["font.family"] = f
            break
        except Exception:
            pass
matplotlib.rcParams["axes.unicode_minus"] = False

# ====== 1) 로드 & 토큰화 ======
df_raw = read_csv_safely(IN_PATH)
text_col = pick_text_column(df_raw)

# 토큰 컬럼이면 그대로 파싱, 아니면 review_text 기반 간이 토큰화
def parse_if_listlike(x):
    if isinstance(x, list): return x
    if isinstance(x, str):
        s = x.strip()
        if (s.startswith("[") and s.endswith("]")) or (s.startswith("(") and s.endswith(")")):
            try:
                obj = eval(s, {"__builtins__": {}})
                if isinstance(obj, (list, tuple)):
                    return [str(t) for t in obj]
            except Exception:
                pass
    return None

docs = []
if re.search(r"(token|morph)", text_col, re.I):
    for s in df_raw[text_col].tolist():
        maybe = parse_if_listlike(s)
        if maybe is None: maybe = simple_ko_tokenize(str(s))
        toks = [t for t in maybe if keep_token(t)]
        if toks: docs.append(toks)
else:
    for s in df_raw[text_col].tolist():
        maybe = simple_ko_tokenize(s)
        toks = [t for t in maybe if keep_token(t)]
        if toks: docs.append(toks)

N = len(docs)
print(f"[INFO] docs={N}")

# ====== 2) DTM ======
df_counter = Counter(); tf_counter = Counter()
for toks in docs:
    df_counter.update(set(toks)); tf_counter.update(toks)

candidates = [(t, c) for t, c in df_counter.items() if c >= MIN_DF]
candidates.sort(key=lambda x: (-x[1], -tf_counter[x[0]]))
vocab = [t for t, _ in candidates[:TOP_V]]
V = len(vocab); vidx = {t: i for i, t in enumerate(vocab)}

DTM = np.zeros((N, V), dtype=np.uint8)
for i, toks in enumerate(docs):
    cols = {vidx[t] for t in toks if t in vidx}
    if cols: DTM[i, list(cols)] = 1

df_vec = DTM.sum(axis=0).astype(int)
tf_vec = np.array([tf_counter[t] for t in vocab], dtype=int)

# ====== 3) PMI/NPMI/PHI ======
co = DTM.T @ DTM
total_docs = float(N)
p_t = df_vec / total_docs
p_xy = co / total_docs
EPS = 1e-12
PMI  = np.log((p_xy + EPS) / (p_t[:, None] * p_t[None, :] + EPS))
NPMI = PMI / (-np.log(p_xy + EPS))
np.fill_diagonal(PMI, 0.0); np.fill_diagonal(NPMI, 0.0)

n11 = co.astype(float)
n1_ = df_vec.astype(float)[:, None]
n_1 = df_vec.astype(float)[None, :]
n00 = N - (n1_ + n_1 - n11)
n10 = n1_ - n11
n01 = n_1 - n11
den = np.sqrt(n1_*(N-n1_)*n_1*(N-n_1)) + 1e-12
PHI = (n11*n00 - n10*n01) / den
np.fill_diagonal(PHI, 0.0)

# ====== 4) PPMI-SVD + KMeans ======
PPMI = np.maximum(PMI, 0.0)
svd = TruncatedSVD(n_components=min(EMB_DIM, max(10, V-1)), random_state=42)
emb = svd.fit_transform(PPMI)

best = {"k": None, "sil": -1, "labels": None}
k_lo, k_hi = K_RANGE
for k in range(k_lo, k_hi+1):
    km = KMeans(n_clusters=k, n_init=10, random_state=42)
    labels = km.fit_predict(emb)
    sample_size = min(SIL_SAMPLE, len(emb))
    try:
        sil = silhouette_score(emb, labels, sample_size=sample_size, random_state=42)
    except Exception:
        sil = -1
    if sil > best["sil"]:
        best = {"k": k, "sil": sil, "labels": labels}

labels = best["labels"]; K = best["k"]
print(f"[CLUSTER] k={K}, silhouette={best['sil']:.3f}")

# ====== 5) 중심성/랭킹 ======
NPMI_pos = np.where(NPMI > 0, NPMI, 0.0)
cent = np.zeros(V, dtype=float)
for ci in range(K):
    idx = np.where(labels == ci)[0]
    if len(idx) == 0: continue
    sub = NPMI_pos[np.ix_(idx, idx)]
    cent[idx] = (sub.sum(axis=1) - np.diag(sub))

c1 = emb[:, 0].astype(float)
svd_c1 = (c1 - c1.min()) / (np.ptp(c1) + 1e-12)
rank_score = 0.45*zscore(cent) + 0.30*zscore(df_vec) + 0.15*zscore(tf_vec) + 0.10*zscore(svd_c1)

token_stats = (
    pd.DataFrame({
        "token": vocab, "df": df_vec, "tf": tf_vec, "cluster": labels,
        "cluster_centrality": cent, "svd_c1": svd_c1, "rank_score": rank_score
    })
    .sort_values(["rank_score","df"], ascending=[False,False])
    .reset_index(drop=True)
)

rows = []
for ci in range(K):
    sub = token_stats[token_stats["cluster"]==ci].head(30)
    rows.append({"cluster": ci, "size": int((labels==ci).sum()),
                 "top_tokens": ", ".join(sub["token"].tolist())})
clusters_df = pd.DataFrame(rows).sort_values("cluster")

# ====== 6) 엣지 ======
pairs = np.triu_indices(V, 1)
npmi_vals = NPMI[pairs]; co_vals = co[pairs]
mask = (npmi_vals > 0) & (co_vals >= MIN_CO_DOCS)
order = np.argsort(-npmi_vals[mask])
i_idx = pairs[0][mask][order]; j_idx = pairs[1][mask][order]

edges_df = pd.DataFrame({
    "token1": [vocab[i] for i in i_idx],
    "token2": [vocab[j] for j in j_idx],
    "co_docs": co_vals[mask][order].astype(int),
    "PMI": PMI[pairs][mask][order],
    "NPMI": npmi_vals[mask][order],
    "PHI": PHI[pairs][mask][order]
})

# ====== 7) Tableau 파일 ======
nodes_tbl = token_stats.copy().rename(columns={
    "token":"node","df":"node_df","tf":"node_tf","cluster":"node_cluster",
    "cluster_centrality":"node_centrality","svd_c1":"node_svd_c1","rank_score":"node_rank"
})
left  = nodes_tbl.add_prefix("src_").rename(columns={"src_node":"token1"})
right = nodes_tbl.add_prefix("dst_").rename(columns={"dst_node":"token2"})
edges_full = edges_df.merge(left, on="token1", how="left").merge(right, on="token2", how="left")

# ====== 8) 저장 ======
stats_path       = OUT_DIR / "morph_stats_kurly_health.csv"
clusters_path    = OUT_DIR / "morph_clusters_kurly_health.csv"
edges_path       = OUT_DIR / "morph_edges_npmi_top_kurly_health.csv"
nodes_path       = OUT_DIR / "tableau_nodes_kurly_health.csv"
edges_full_path  = OUT_DIR / "tableau_edges_full_kurly_health.csv"

token_stats.to_csv(stats_path, index=False, encoding="utf-8-sig")
clusters_df.to_csv(clusters_path, index=False, encoding="utf-8-sig")
edges_df.to_csv(edges_path, index=False, encoding="utf-8-sig")
nodes_tbl.to_csv(nodes_path, index=False, encoding="utf-8-sig")
edges_full.to_csv(edges_full_path, index=False, encoding="utf-8-sig")

print("[SAVE]")
for p in [stats_path, clusters_path, edges_path, nodes_path, edges_full_path]:
    print(" -", p)

# ====== 9) 시각화 ======
def savefig(path, tight=True, dpi=200):
    if tight: plt.tight_layout()
    plt.savefig(path, dpi=dpi)
    plt.close()

HEAT_TOP_EFF = min(HEAT_TOP, len(token_stats))
sel_tokens = token_stats["token"].head(HEAT_TOP_EFF).tolist()
sel_idx = [vocab.index(t) for t in sel_tokens]
mat = np.where(NPMI>0, NPMI, 0.0)[np.ix_(sel_idx, sel_idx)]

plt.figure(figsize=(9,7))
plt.imshow(mat, interpolation="nearest")
plt.xticks(range(HEAT_TOP_EFF), sel_tokens, rotation=90)
plt.yticks(range(HEAT_TOP_EFF), sel_tokens)
plt.title("NPMI Heatmap (kurly_health)")
plt.colorbar()
savefig(OUT_DIR / "npmi_heatmap_top25_kurly_health.png")

tp = edges_df.head(PAIR_TOP)
plt.figure(figsize=(10,7))
y = [f"{a}-{b}" for a,b in zip(tp["token1"], tp["token2"])]
plt.barh(range(len(y)), tp["NPMI"].values)
plt.yticks(range(len(y)), y)
plt.xlabel("NPMI")
plt.title("Top NPMI Pairs (kurly_health)")
savefig(OUT_DIR / "top_pairs_npmi_bar_kurly_health.png")

plt.figure(figsize=(9,7))
xv, yv = emb[:,0], emb[:,1]
plt.scatter(xv, yv, c=labels, s=22)
for ci in range(K):
    sub = token_stats[token_stats["cluster"]==ci].head(3)
    for tok in sub["token"]:
        idx = vocab.index(tok)
        plt.text(xv[idx], yv[idx], tok, fontsize=8)
plt.title(f"SVD Embedding Scatter (kurly_health, k={K})")
plt.xlabel("SVD-1"); plt.ylabel("SVD-2")
savefig(OUT_DIR / "cluster_scatter_svd_kurly_health.png")

plt.figure(figsize=(9,7))
plt.scatter(token_stats["df"], token_stats["rank_score"], s=18)
for tok in token_stats.head(20)["token"]:
    ridx = token_stats.index[token_stats["token"]==tok][0]
    plt.text(token_stats.loc[ridx, "df"], token_stats.loc[ridx, "rank_score"], tok, fontsize=8)
plt.xlabel("DF (문서수)")
plt.ylabel("Rank Score")
plt.title("랭킹 vs DF (kurly_health)")
savefig(OUT_DIR / "rank_vs_df_scatter_kurly_health.png")

print("[DONE] 모든 결과가 utf-8-sig로 저장되었습니다.")


[INFO] docs=866
[CLUSTER] k=7, silhouette=0.259
[SAVE]
 - kurly_health_morph\morph_stats_kurly_health.csv
 - kurly_health_morph\morph_clusters_kurly_health.csv
 - kurly_health_morph\morph_edges_npmi_top_kurly_health.csv
 - kurly_health_morph\tableau_nodes_kurly_health.csv
 - kurly_health_morph\tableau_edges_full_kurly_health.csv
[DONE] 모든 결과가 utf-8-sig로 저장되었습니다.
